In [1]:
import pandas as pd

## **Sreenivas+ 2026**

### Data Description

### CDS X-Matching

Read-in xmatch.

In [2]:
array_full = pd.read_csv("../0_inputdata_CDSxmatch/Sree2026_cdsxmatch.csv", low_memory=False)

Remove flagged entries.

In [3]:
array_full = array_full[
    (array_full["f_deltanu"] == 1) &
    (array_full["blendFlag"] == 1) 
]
print(f"Cleaned data length: {len(array_full):.0f}")

Cleaned data length: 9982


Convert EV state, with:

 - 0 ==> not specified
 - 1 ==> likely RGB
 - 2 ==> likely CHeB

In [4]:
array_full["State"] = array_full["State"].replace({
    0.0: int(0),
    1.0: int(1),
    2.0: int(2)
})

Curate simplfied array.

In [5]:
s26 = pd.DataFrame({
    # star identifiers
    "gaiadr3_id": array_full["Source"],
    "ra": array_full["RAdeg"],
    "dec": array_full["DEdeg"],
    # asteroseismic parameters
    "numax": array_full["numax"],
    "e_numax": array_full["e_numax"],
    "dnu": array_full["deltanu"],
    "e_dnu": array_full["e_deltanu"],
    "ev_state": array_full["State"],
    # Gaia parameters
    "GAIA-teff": array_full["Teff"],
    "GAIA-e_teff": (array_full["B_Teff"] - array_full["b_Teff"]) / 2,
    "GAIA-logg": array_full["logg"],
    "GAIA-e_logg": (array_full["B_logg"] - array_full["b_logg"]) / 2,
    "GAIA-fe_h": array_full["[Fe/H]"],
    "GAIA-e_fe_h": (array_full["B_[Fe/H]"] - array_full["b_[Fe/H]"]) / 2,
    # GSP Spec parameters
    "GSPSPEC-teff": array_full["Teff-S"],
    "GSPSPEC-e_teff": (array_full["B_Teff-S"] - array_full["b_Teff-S"]) / 2,
    "GSPSPEC-logg": array_full["logg-S"],
    "GSPSPEC-e_logg": (array_full["B_logg-S"] - array_full["b_logg-S"]) / 2,
    "GSPSPEC-fe_h": array_full["[Fe/H]-S"],
    "GSPSPEC-e_fe_h": (array_full["B_[Fe/H]-S"] - array_full["b_[Fe/H]-S"]) / 2,
    "GSPSPEC-si_fe": array_full["[Si/Fe]-S"],
    "GSPSPEC-e_si_fe": (array_full["B_[Si/Fe]-S"] - array_full["b_[Si/Fe]-S"]) / 2,
    "GSPSPEC-ca_fe": array_full["[Ca/Fe]-S"],
    "GSPSPEC-e_ca_fe": (array_full["B_[Ca/Fe]-S"] - array_full["b_[Ca/Fe]-S"]) / 2,
    "GSPSPEC-ti_fe": array_full["[Ti/Fe]-S"],
    "GSPSPEC-e_ti_fe": (array_full["B_[Ti/Fe]-S"] - array_full["b_[Ti/Fe]-S"]) / 2,
    "GSPSPEC-mg_fe": array_full["[Mg/Fe]-S"],
    "GSPSPEC-e_mg_fe": (array_full["B_[Mg/Fe]-S"] - array_full["b_[Mg/Fe]-S"]) / 2,
    "GSPSPEC-nd_fe": array_full["[Nd/Fe]-S"],
    "GSPSPEC-e_nd_fe": (array_full["B_[Nd/Fe]-S"] - array_full["b_[Nd/Fe]-S"]) / 2,
    "GSPSPEC-s_fe": array_full["[S/Fe]-S"],
    "GSPSPEC-e_s_fe": (array_full["B_[S/Fe]-S"] - array_full["b_[S/Fe]-S"]) / 2,
    "GSPSPEC-zr_fe": array_full["[Zr/Fe]-S"],
    "GSPSPEC-e_zr_fe": (array_full["B_[Zr/Fe]-S"] - array_full["b_[Zr/Fe]-S"]) / 2,
    "GSPSPEC-n_fe": array_full["[N/Fe]-S"],
    "GSPSPEC-e_n_fe": (array_full["B_[N/Fe]-S"] - array_full["b_[N/Fe]-S"]) / 2,
    "GSPSPEC-cr_fe": array_full["[Cr/Fe]-S"],
    "GSPSPEC-e_cr_fe": (array_full["B_[Cr/Fe]-S"] - array_full["b_[Cr/Fe]-S"]) / 2,
    "GSPSPEC-ce_fe": array_full["[Ce/Fe]-S"],
    "GSPSPEC-e_ce_fe": (array_full["B_[Ce/Fe]-S"] - array_full["b_[Ce/Fe]-S"]) / 2,
    "GSPSPEC-ni_fe": array_full["[Ni/Fe]-S"],
    "GSPSPEC-e_ni_fe": (array_full["B_[Ni/Fe]-S"] - array_full["b_[Ni/Fe]-S"]) / 2
})

s26 = s26.dropna(subset=["e_numax", "e_dnu"])

Taken within $0.75\times$ and $1.35\times$ of $\Delta\nu$-$\nu_\mathrm{max}$ relation (Stello+ 09, Mosser+ 11):

$$ \Delta\nu = 0.27 \nu_\mathrm{max}^{0.76}. $$

In [6]:
def relation(numax):
    coeff = 0.27
    return coeff * numax**(0.76)

mask = (s26["dnu"] < 1.35*relation(s26["numax"])) & (s26["dnu"] > 0.75*relation(s26["numax"]))

s26 = s26[mask]

Write to `.csv` file.

In [7]:
s26.to_csv("./sree26_asteroseismic.csv", index=False)